# Phase 1 - Parsing the SemEval-2013 DDI Corpus
## Introduction

This notebook focuses on transforming the raw SemEval-2013 Drug–Drug Interaction (DDI) dataset from its original XML format into a structured and analysis-ready representation. The dataset contains sentence-level annotations describing drug entities and their interactions, which must be systematically extracted and standardised for downstream machine learning tasks.

The primary objective is to parse XML files to accurately extract sentences, drug mentions, and interaction pairs while ensuring robustness against malformed or inconsistent data. The extracted information is then organised into a clean tabular format, preserving essential metadata such as corpus source (DrugBank or MedLine) and offset validity for entity spans. To ensure data integrity, validation steps are performed to detect label inconsistencies, missing values, and offset mismatches.

In addition, exploratory analysis is conducted to understand dataset characteristics, including class distribution, imbalance across interaction types, and differences between corpus sources. The final output of this notebook is a set of structured CSV files for training and testing, providing a reliable and reproducible foundation for subsequent stages such as feature engineering and model development.

## Environment Setup and Dependency Initialisation

This section sets up the execution environment by importing all required libraries for file handling, XML parsing, data processing, and statistical analysis. It also configures the system path to access the shared `config.py` file, ensuring consistent use of project-level settings. To maintain reproducibility, warnings are suppressed and a fixed random seed is applied. Finally, verification messages confirm that the environment has been correctly initialised.

In [1]:
# File System Operations
import os

# CSV Handling
import csv

# XML Parsing
from xml.etree import ElementTree as ET

# Statistics and Counting
from collections import Counter

# Data Manipulation
import pandas as pd

# Typing
from typing import Dict, List, Tuple

# Path Setup for Config Import
import sys
sys.path.insert(0, os.path.abspath('../'))

from config import *

# Additional Imports
import random
import warnings

# Configuration Setup
warnings.filterwarnings('ignore')
random.seed(RANDOM_STATE)

## Raw Data Directory Verification

This section verifies that the required training and test data directories are accessible before parsing begins. The paths are retrieved from the central configuration file, ensuring consistency across the project. By checking their existence early, potential issues such as missing or incorrect paths can be identified and resolved, preventing failures in later stages of the pipeline.

In [2]:
# Raw Data Directory Verification
for path in [str(RAW_TRAIN_DIR), str(RAW_TEST_DIR)]:
    status = 'FOUND' if os.path.exists(path) else 'NOT FOUND. check your path!'

## DDI XML File Parsing Function

This section defines a function to parse individual SemEval XML files and convert them into a structured format suitable for analysis. The function extracts sentence-level information, identifies drug entities, and constructs interaction pairs while preserving important metadata.

The parsing process begins with robust error handling to ensure that malformed or corrupted XML files do not interrupt execution. Each file is also classified by its source (DrugBank or MedLine) based on its path, enabling later analysis of corpus-specific patterns.

Within each sentence, drug entities are extracted along with their text, type, and character offsets. These offsets are validated by comparing the annotated spans with the actual sentence text, and any mismatches are recorded for inspection. This step is important because discontinuous entity spans are common in the dataset and can affect downstream processing.

The function then iterates over all annotated drug pairs and assigns interaction labels based on the DDI flag and type. Labels are normalised to ensure consistency, and invalid or incomplete pairs are safely ignored. The final output is a structured list of interaction records, along with logs for offset inconsistencies and parsing errors, providing both usable data and diagnostic information.

In [3]:
# DDI XML File Parsing Function

def parse_ddi_file(
    filepath: str,
) -> Tuple[List[dict], List[dict], List[dict]]:

    rows: List[dict] = []
    offset_warnings: List[dict] = []
    parse_errors: List[dict] = []

    # XML Parsing with Error Handling
    try:
        tree = ET.parse(filepath)
        root = tree.getroot()
    except ET.ParseError as e:
        parse_errors.append({'file': filepath, 'error': str(e)})
        print(f'Skip Malformed XML: {os.path.basename(filepath)} {e}')
        return rows, offset_warnings, parse_errors
    except Exception as e:
        parse_errors.append({'file': filepath, 'error': str(e)})
        print(f'Skip Unexpected error: {os.path.basename(filepath)} {e}')
        return rows, offset_warnings, parse_errors

    # Determine Corpus Source
    norm_path = filepath.replace(os.sep, '/')
    if 'DrugBank' in norm_path:
        corpus_source = 'DrugBank'
    elif 'MedLine' in norm_path:
        corpus_source = 'MedLine'
    else:
        corpus_source = 'Unknown'

    # Sentence-Level Processing
    for sentence in root.iter('sentence'):
        sentence_id = sentence.get('id', '')
        sentence_text = sentence.get('text', '')

        # Entity Extraction
        entities: Dict[str, dict] = {}
        for entity in sentence.findall('entity'):
            eid = entity.get('id')
            raw_offset = entity.get('charOffset', '')
            entity_text = entity.get('text', '')
            entity_type = entity.get('type', 'unknown')

            offset_valid = True
            try:
                first_span = raw_offset.split(';')[0]
                start, end = map(int, first_span.split('-'))
                extracted = sentence_text[start:end + 1]

                if extracted.lower() != entity_text.lower():
                    offset_valid = False
                    offset_warnings.append({
                        'sentence_id' : sentence_id,
                        'entity_id' : eid,
                        'expected' : entity_text,
                        'extracted' : extracted,
                        'char_offset' : raw_offset,
                    })
            except Exception:
                offset_valid = False

            entities[eid] = {
                'text' : entity_text,
                'type' : entity_type,
                'charOffset' : raw_offset,
                'offset_valid' : offset_valid,
            }

        # Pair Extraction
        for pair in sentence.findall('pair'):
            e1_id = pair.get('e1')
            e2_id = pair.get('e2')
            ddi_flag = pair.get('ddi', 'false').lower()
            ddi_type = pair.get('type', '')

            e1 = entities.get(e1_id, {})
            e2 = entities.get(e2_id, {})

            # Label Assignment
            if ddi_flag == 'true' and ddi_type:
                label = ddi_type.lower()
            elif ddi_flag == 'true':
                label = 'int'
            else:
                label = 'false'

            # Skip invalid pairs
            if not e1 or not e2:
                continue

            rows.append({
                'sentence_id' : sentence_id,
                'sentence_text' : sentence_text,
                'drug1_id' : e1_id,
                'drug1_text' : e1.get('text', ''),
                'drug1_type' : e1.get('type', 'unknown'),
                'drug2_id' : e2_id,
                'drug2_text' : e2.get('text', ''),
                'drug2_type' : e2.get('type', 'unknown'),
                'ddi' : ddi_flag == 'true',
                'interaction_type' : label,
                'corpus_source' : corpus_source,
                'offset_valid' : e1.get('offset_valid', True) and e2.get('offset_valid', True),
                'source_file' : os.path.basename(filepath),
            })

    return rows, offset_warnings, parse_errors

## DDI Folder Parsing and Aggregation

This section defines a function to parse all XML files within a given directory and aggregate the extracted drug–drug interaction data. It builds upon the single-file parser to process the entire dataset in a consistent and scalable manner.

The function first scans the directory recursively to collect all XML files, ensuring that nested folder structures are also handled. Each file is then parsed individually using the previously defined parsing function, and the results are accumulated into unified lists containing all interaction rows, offset warnings, and parsing errors.

During processing, the function tracks sentence frequencies to provide insight into how often the same sentence appears across multiple drug pairs. This is expected behaviour in the SemEval dataset, as a single sentence can generate multiple interaction pairs.

The function also generates informative reports, including parsing failures and character offset mismatches. These diagnostics help identify potential data quality issues while confirming that most inconsistencies arise from known annotation patterns such as discontinuous spans.

Overall, this step produces a complete, structured representation of the dataset along with useful validation insights, preparing the data for further analysis and modelling.

In [4]:
# DDI Folder Parsing and Aggregation

def parse_ddi_folder(data_dir: str) -> Tuple[List[dict], List[dict]]:

    all_rows: List[dict] = []
    all_warnings: List[dict] = []
    all_parse_errors: List[dict] = []
    xml_files: List[str] = []
    sentence_counter = Counter()

    # Collect XML File Paths
    for root_dir, _, files in os.walk(data_dir):
        for fname in files:
            if fname.endswith('.xml'):
                xml_files.append(os.path.join(root_dir, fname))

    # Parse Each File
    for fpath in sorted(xml_files):
        rows, warnings, errors = parse_ddi_file(fpath)

        all_rows.extend(rows)
        all_warnings.extend(warnings)
        all_parse_errors.extend(errors)

        # Sentence Frequency Tracking
        for r in rows:
            sentence_counter[r['sentence_text'].strip()] += 1

        status = f'{len(rows):4d} pairs'
        print(f'{os.path.basename(fpath):50s} to {status}')

    # Parsing Error Report
    if all_parse_errors:
        print(f'\n[WARNING] {len(all_parse_errors)} file(s) failed to parse:')
        for err in all_parse_errors:
            print(f'{err["file"]} : {err["error"]}')
    else:
        print('\nAll files parsed successfully (no errors).')

    # Offset Validation Report
    if all_warnings:
        print(f'\n[INFO] {len(all_warnings)} charOffset mismatch(es) found.')
        print('These are expected for discontinuous multi-word entities (semi-colon offsets).')
        print('Affected rows have offset_valid=False in the output CSV.')
        print('First 3 examples:')
        for w in all_warnings[:3]:
            print(f'  Sentence {w["sentence_id"]}: '
                  f'expected "{w["expected"]}" '
                  f'but first-span gives "{w["extracted"]}" '
                  f'at offset {w["char_offset"]}')
    else:
        print('All charOffsets validated successfully.')

    # Sentence Repetition Report
    duplicated = {s: c for s, c in sentence_counter.items() if c > 1}

    print(f'\n[INFO] Sentence repetition report (informational only):')
    print(f'Total unique sentence texts : {len(sentence_counter)}')
    print(f'Sentences appearing >1x : {len(duplicated)}')
    print(f'(Each repetition = one additional drug pair from the same sentence — expected)')

    if duplicated:
        print('Top 5 most repeated sentences (pair count):')
        for sent, count in sorted(duplicated.items(), key=lambda x: -x[1])[:5]:
            preview = sent[:70] + '...' if len(sent) > 70 else sent
            print(f'({count}x) {preview}')

    return all_rows, all_parse_errors

## Dataset Summary and Export Utilities

This section defines utility functions for analysing the parsed dataset and exporting it to a structured CSV file. The `print_stats` function provides an overview of the dataset by reporting the total number of interaction pairs, the distribution of interaction types, and the breakdown across corpus sources. It also computes the number of positive and negative DDI instances, along with the imbalance ratio, which is important for understanding dataset skew and guiding model selection.

The `save_csv` function handles exporting the processed data into a consistent tabular format. Before saving, it performs a final validation check to ensure that all interaction labels conform to the predefined set, flagging any unexpected values. The function then writes the data with a fixed column structure, ensuring reproducibility and compatibility with downstream tasks such as feature engineering and model training.

In [5]:
# Dataset Statistics Function

def print_stats(rows: List[dict]) -> None:

    print('\n Dataset statistics')
    print(f'Total pairs : {len(rows)}')

    label_counts  = Counter(r['interaction_type'] for r in rows)
    source_counts = Counter(r.get('corpus_source', 'Unknown') for r in rows)

    ddi_true  = sum(1 for r in rows if r['ddi'] is True)
    ddi_false = sum(1 for r in rows if r['ddi'] is False)

    print(f'DDI = true : {ddi_true:6d}')
    print(f'DDI = false : {ddi_false:6d}')

    if ddi_true > 0:
        ratio = ddi_false / ddi_true
        print(f'Imbalance ratio (neg:pos) : {ratio:.1f}:1 important for model selection')

    print('\nPairs by corpus source:')
    for src, count in source_counts.most_common():
        print(f'  {src:<12} {count:6d}')

    print('\nInteraction type breakdown:')
    for label, count in label_counts.most_common():
        bar = chr(9608) * (count // 50)
        print(f'{label:<15} {count:5d}  {bar}')


# CSV Export Function

def save_csv(rows: List[dict], output_path: str) -> None:

    if not rows:
        print('No data to save.')
        return

    # Final Label Guard
    unexpected_labels = {r['interaction_type'] for r in rows} - VALID_LABELS
    if unexpected_labels:
        print(f'[WARNING] Unexpected labels found before save: {unexpected_labels}')
        print('Review parse_ddi_file — these rows will still be written.')

    os.makedirs(os.path.dirname(os.path.abspath(output_path)), exist_ok=True)

    fieldnames = [
        'sentence_id', 'sentence_text',
        'drug1_id', 'drug1_text', 'drug1_type',
        'drug2_id', 'drug2_text', 'drug2_type',
        'ddi', 'interaction_type',
        'corpus_source',
        'offset_valid',
        'source_file',
    ]

    with open(output_path, 'w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)

    print(f'Saved {len(rows)} rows to {output_path}')

## Training Data Parsing and Export

This section executes the complete data processing pipeline for the training dataset. It begins by parsing all XML files from the training directory, converting them into structured interaction records. The parsed data is then analysed using summary statistics to understand class distribution and dataset characteristics. Finally, the processed data is saved to a CSV file, ensuring it is ready for subsequent stages such as feature engineering and model development.

In [6]:
# Training Data Parsing and Export

print('-' * 28)
print('Parsing TRAINING data')
print('-' * 28)

# Parse Training Data
train_rows, train_parse_errors = parse_ddi_folder(str(RAW_TRAIN_DIR))

# Display Statistics
print_stats(train_rows)

# Save to CSV
save_csv(train_rows, TRAIN_CSV)

----------------------------
Parsing TRAINING data
----------------------------
19-norandrostenedione_ddi.xml                      to    0 pairs
Abarelix_ddi.xml                                   to    1 pairs
Abatacept_ddi.xml                                  to   81 pairs
Abciximab_ddi.xml                                  to   35 pairs
Acamprosate_ddi.xml                                to   31 pairs
Acarbose_ddi.xml                                   to  115 pairs
Acebutolol_ddi.xml                                 to   27 pairs
Acetazolamide_ddi.xml                              to   16 pairs
Acetohydroxamic Acid_ddi.xml                       to    1 pairs
Aciclovir_ddi.xml                                  to    1 pairs
Acitretin_ddi.xml                                  to   55 pairs
Adalimumab_ddi.xml                                 to   14 pairs
Adapalene_ddi.xml                                  to    6 pairs
Adefovir Dipivoxil_ddi.xml                         to   36 pairs
Adenosine_

## Test Data Parsing and Export

This section applies the same processing pipeline to the test dataset to ensure consistency with the training data. All XML files in the test directory are parsed into structured interaction records, followed by computation of summary statistics to verify distribution and integrity. The processed data is then saved as a CSV file, making it ready for evaluation and comparison with model predictions.

In [7]:
# Test Data Parsing and Export

print('-' * 28)
print('Parsing TEST data')
print('-' * 28)

test_rows, test_parse_errors = parse_ddi_folder(RAW_TEST_DIR)

print_stats(test_rows)

save_csv(test_rows, TEST_CSV)

----------------------------
Parsing TEST data
----------------------------
Abacavir.xml                                       to   23 pairs
Ezetimibe.xml                                      to   52 pairs
Fluconazole.xml                                    to  216 pairs
Mazindol.xml                                       to  176 pairs
Mebendazole.xml                                    to    3 pairs
Mecamylamine.xml                                   to    6 pairs
Medroxyprogesterone.xml                            to    3 pairs
Megestrol.xml                                      to   13 pairs
Melatonin.xml                                      to  136 pairs
Meloxicam.xml                                      to   63 pairs
Melphalan.xml                                      to    7 pairs
Mepenzolate.xml                                    to  110 pairs
Meperidine.xml                                     to   55 pairs
Mephenytoin.xml                                    to   28 pairs
Mequitazine.xm

## Processed Dataset Loading and Inspection

This section loads the processed training and test datasets into pandas DataFrames and performs an initial inspection. It begins by displaying the shape and column structure of both datasets to verify that the data has been correctly parsed and stored.

The distribution of the binary DDI label in the training set is then examined to provide a quick overview of class balance. Finally, a small preview of the dataset is displayed to confirm that the fields, values, and overall structure are as expected before proceeding to further analysis.

In [8]:
# Load Training Data
train_df = pd.read_csv(TRAIN_CSV)
test_df = pd.read_csv(TEST_CSV)

# Dataset Overview
print(f'Training set shape : {train_df.shape}')
print(f'Test set shape : {test_df.shape}')
print()

# DDI Column Inspection
print('DDI value distribution (train):')
print(train_df['ddi'].value_counts().to_string())
print()

# Preview
print('First 3 rows:')
train_df.head(3)

Training set shape : (27792, 13)
Test set shape : (6657, 13)

DDI value distribution (train):
ddi
False    23771
True      4021

First 3 rows:


,sentence_id,sentence_text,drug1_id,drug1_text,drug1_type,drug2_id,drug2_text,drug2_type,ddi,interaction_type,corpus_source,offset_valid,source_file
0,DDI-DrugBank.d519.s3,Laboratory Tests Response to Plenaxis should b...,DDI-DrugBank.d519.s3.e0,Plenaxis,brand,DDI-DrugBank.d519.s3.e1,testosterone,drug,False,false,DrugBank,True,Abarelix_ddi.xml
1,DDI-DrugBank.d297.s1,Population pharmacokinetic analyses revealed t...,DDI-DrugBank.d297.s1.e0,MTX,drug,DDI-DrugBank.d297.s1.e1,NSAIDs,group,False,false,DrugBank,True,Abatacept_ddi.xml
2,DDI-DrugBank.d297.s1,Population pharmacokinetic analyses revealed t...,DDI-DrugBank.d297.s1.e0,MTX,drug,DDI-DrugBank.d297.s1.e2,corticosteroids,group,False,false,DrugBank,True,Abatacept_ddi.xml


## Sample Inspection by Interaction Type

This section displays one representative example for each interaction type to validate the correctness of the parsed data. For each class, a sample sentence is shown along with the corresponding drug entities, their types, corpus source, and offset validation status.

This inspection helps ensure that the labels are properly aligned with the sentence context and that entity extraction has been performed correctly. It also provides an intuitive understanding of how different interaction types appear in the dataset before proceeding to modelling.

In [9]:
# Sample Inspection by Interaction Type

for label in ['mechanism', 'effect', 'advise', 'int', 'false']:
    subset = train_df[train_df['interaction_type'] == label]
    if subset.empty:
        print(f'[{label.upper()}] No examples found.\n')
        continue

    row = subset.iloc[0]
    print(f'[{label.upper()}]')
    print(f'Sentence : {row["sentence_text"][:120]}...' if len(row['sentence_text']) > 120 else f'Sentence : {row["sentence_text"]}')
    print(f'Drug 1 : {row["drug1_text"]} ({row["drug1_type"]})')
    print(f'Drug 2 : {row["drug2_text"]} ({row["drug2_type"]})')
    print(f'Corpus : {row["corpus_source"]}')
    print(f'Offset OK : {row["offset_valid"]}')
    print()

[MECHANISM]
Sentence : Co-administration of naltrexone with Acamprosate produced a 25% increase in AUC and a 33% increase in the Cmax of acampr...
Drug 1 : naltrexone (drug)
Drug 2 : Acamprosate (drug)
Corpus : DrugBank
Offset OK : True

[EFFECT]
Sentence : Concurrent administration of a TNF antagonist with ORENCIA has been associated with an increased risk of serious infecti...
Drug 1 : TNF antagonist (group)
Drug 2 : ORENCIA (brand)
Corpus : DrugBank
Offset OK : True

[ADVISE]
Sentence : Concurrent therapy with ORENCIA and TNF antagonists is not recommended.
Drug 1 : ORENCIA (brand)
Drug 2 : TNF antagonists (group)
Corpus : DrugBank
Offset OK : True

[INT]
Sentence : Drugs and other substances demonstrated to be CYP 3A inhibitors on the basis of clinical studies involving benzodiazepin...
Drug 1 : alprazolam (drug)
Drug 2 : diltiazem (drug)
Corpus : DrugBank
Offset OK : True

[FALSE]
Sentence : Laboratory Tests Response to Plenaxis should be monitored by measuring serum total testost

## Corpus Distribution and Offset Validation Analysis

This section analyses how interaction types are distributed across different corpus sources and evaluates the quality of entity offset annotations. It first computes the count of each interaction type within each corpus, followed by a percentage-based view to better understand relative distributions and potential dataset bias.

The section also examines the validity of character offsets used for entity extraction. It reports the number of valid and invalid offsets, along with the percentage of mismatches. Since discontinuous entity spans are common in the dataset, a small proportion of invalid offsets is expected. Sample rows with invalid offsets are displayed to provide insight into these cases and confirm that they do not indicate critical parsing errors.

In [10]:
# Class Distribution by Corpus Source (Counts)

print('Class distribution by corpus source (counts) \n')
pivot = train_df.groupby(['corpus_source', 'interaction_type']).size().unstack(fill_value=0)
print(pivot.to_string())


# Percentage Distribution

print('\nClass distribution by corpus source (row %)\n')
pivot_pct = pivot.div(pivot.sum(axis=1), axis=0).mul(100).round(1)
print(pivot_pct.to_string())


# Offset Validation Analysis

print('\n Offset validity summary')
invalid = train_df[train_df['offset_valid'] == False]

print(f'Total rows : {len(train_df)}')
print(f'Offset valid : {(train_df["offset_valid"] == True).sum()}')
print(f'Offset invalid : {len(invalid)}')

if len(invalid) > 0:
    pct = len(invalid) / len(train_df) * 100
    print(f'Invalid % : {pct:.2f}% (expected due to discontinuous entity spans)')
    print('\nFirst 3 invalid rows:')
    print(invalid[['sentence_id', 'drug1_text', 'drug2_text', 'interaction_type']].head(3).to_string())

Class distribution by corpus source (counts) 

interaction_type  advise  effect  false  int  mechanism
corpus_source                                          
DrugBank             818    1535  22216  179       1257
MedLine                8     152   1555   10         62

Class distribution by corpus source (row %)

interaction_type  advise  effect  false  int  mechanism
corpus_source                                          
DrugBank             3.1     5.9   85.4  0.7        4.8
MedLine              0.4     8.5   87.0  0.6        3.5

 Offset validity summary
Total rows : 27792
Offset valid : 27663
Offset invalid : 129
Invalid % : 0.46% (expected due to discontinuous entity spans)

First 3 invalid rows:
               sentence_id                             drug1_text                         drug2_text interaction_type
116   DDI-DrugBank.d532.s4       diagnostic monoclonal antibodies  therapeutic monoclonal antibodies            false
1169  DDI-DrugBank.d356.s4  non-steroidal anti-inf

## Data Quality Validation Checks

This section performs a series of validation checks to ensure the integrity and reliability of the processed dataset before proceeding further. It verifies that there are no missing values in critical fields such as sentence text, drug names, and interaction labels, and confirms that all labels conform to the predefined valid set.

The checks also identify empty sentences or missing drug entities, which could negatively impact model performance. In addition, the dataset is examined for any unknown corpus sources and parsing failures, providing insight into potential data inconsistencies. A validation is also performed to ensure that no self-pair interactions exist, as these are not valid in the SemEval annotations.

Finally, the section compares label distributions between the training and test sets to ensure consistency. A summary message is displayed at the end to indicate whether the dataset passes all checks or requires further review.

In [11]:
# Data Quality Validation Checks
print('-' * 28)
print('Quality checks\n')
print('-' * 28)

passed = True

# Missing Values Check
critical_cols = ['sentence_text', 'drug1_text', 'drug2_text', 'interaction_type']
missing = train_df[critical_cols].isnull().sum()

print('1. Missing values in critical columns:')
print(missing.to_string())
if missing.sum() > 0:
    print('FAIL'); passed = False
else:
    print('')


# Label Validation
actual_labels = set(train_df['interaction_type'].unique())
unexpected = actual_labels - VALID_LABELS

print(f'2. Unexpected labels: {unexpected if unexpected else "none"}')
if unexpected:
    print('FAIL'); passed = False
else:
    print('')


# Empty Sentences Check
empty_sents = (train_df['sentence_text'].str.strip() == '').sum()
print(f'3. Empty sentences: {empty_sents}')
print('' if empty_sents == 0 else 'FAIL'); passed = passed and (empty_sents == 0)


# Empty Drug Names Check
empty_drugs = (
    (train_df['drug1_text'].str.strip() == '') |
    (train_df['drug2_text'].str.strip() == '')
).sum()
print(f'4. Rows with empty drug names: {empty_drugs}')
print('' if empty_drugs == 0 else 'FAIL'); passed = passed and (empty_drugs == 0)


# Corpus Source Validation
unknown_source = (train_df['corpus_source'] == 'Unknown').sum()
print(f'5. Rows with unknown corpus source: {unknown_source}')
print('' if unknown_source == 0 else 'WARN some files could not be classified')


# Parsing Errors Check
n_errors = len(train_parse_errors)
print(f'6. Files that failed to parse: {n_errors}')
print('' if n_errors == 0 else f'WARN {n_errors} file(s) skipped')


# Self-Pair Detection
self_pairs = (train_df['drug1_id'] == train_df['drug2_id']).sum()
print(f'7. Self-pair rows (drug1_id == drug2_id): {self_pairs}')
if self_pairs > 0:
    print('FAIL — self-pairs should not exist in SemEval annotations'); passed = False
else:
    print('')


# Train/Test Label Consistency
train_labels = set(train_df['interaction_type'].unique())
test_labels = set(test_df['interaction_type'].unique())
label_diff = train_labels.symmetric_difference(test_labels)
print(f'8. Train/test label set mismatch: {label_diff if label_diff else "none"}')
print('' if not label_diff else 'WARN — label sets differ between train and test')


# Final Summary
print('All checks passed' if passed else 'Issues found. Review the above messages before proceeding.')

----------------------------
Quality checks

----------------------------
1. Missing values in critical columns:
sentence_text       0
drug1_text          0
drug2_text          0
interaction_type    0

2. Unexpected labels: none

3. Empty sentences: 0

4. Rows with empty drug names: 0

5. Rows with unknown corpus source: 0

6. Files that failed to parse: 0

7. Self-pair rows (drug1_id == drug2_id): 0

8. Train/test label set mismatch: none

All checks passed


## Consolidated Label Distribution Summary

This section summarises the distribution of interaction labels for both the training and test datasets. It reports the count and percentage of each interaction type, providing a clear view of how classes are represented within each split.

In addition, the section analyses the binary DDI distribution by reporting the number of positive and negative samples. The imbalance ratio between negative and positive instances is also computed, which is important for understanding dataset skew and guiding model selection and evaluation strategies. Comparing these statistics across both datasets helps ensure consistency and reliability in subsequent modelling.

In [12]:
# Consolidated Label Distribution Summary

for split_name, df in [('TRAIN', train_df), ('TEST', test_df)]:
    total  = len(df)
    counts = df['interaction_type'].value_counts()
    pcts = (counts / total * 100).round(2)

    summary = pd.DataFrame({
        'Count' : counts,
        'Percentage' : pcts,
    })
    summary.index.name = 'Label'

    print(f'\n--- {split_name} SET (n={total}) ---')
    print(summary.to_string())
    print()

    pos = df[df['ddi'] == True]
    neg = df[df['ddi'] == False]
    print(f'Positive (DDI=True) : {len(pos):5d} ({len(pos)/total*100:.1f}%)')
    print(f'Negative (DDI=False) : {len(neg):5d} ({len(neg)/total*100:.1f}%)')
    if len(pos) > 0:
        print(f'Imbalance ratio : {len(neg)/len(pos):.1f}:1')


--- TRAIN SET (n=27792) ---
           Count  Percentage
Label                       
false      23771       85.53
effect      1687        6.07
mechanism   1319        4.75
advise       826        2.97
int          189        0.68

Positive (DDI=True) :  4021 (14.5%)
Negative (DDI=False) : 23771 (85.5%)
Imbalance ratio : 5.9:1

--- TEST SET (n=6657) ---
           Count  Percentage
Label                       
false       5678       85.29
effect       360        5.41
mechanism    302        4.54
advise       221        3.32
int           96        1.44

Positive (DDI=True) :   979 (14.7%)
Negative (DDI=False) :  5678 (85.3%)
Imbalance ratio : 5.8:1


## Summary

In this notebook, the SemEval DDI dataset was successfully parsed and transformed into a structured format suitable for analysis and modelling.

Key outcomes of this phase include:

- **Robust Parsing Pipeline**
  - Extracted drug entities and interaction pairs from XML files
  - Handled parsing errors without interrupting execution

- **Structured Dataset Creation**
  - Converted raw data into consistent tabular format
  - Saved processed training and test datasets as CSV files

- **Data Quality Assurance**
  - Performed checks for missing values, invalid labels, and self-pairs
  - Analysed offset mismatches and confirmed expected behaviour

- **Dataset Insights**
  - Evaluated class distribution and imbalance
  - Compared training and test set consistency

Overall, this notebook establishes a clean, validated, and well-understood dataset, ensuring readiness for the next stage of feature engineering and model development.